# Entrenamiento y publicación de versiones

Este notebook corre dentro del contenedor **jupyter**. Todo lo que se guarde en
`MODELS_DIR` (`/models`) queda en el volumen compartido y lo ve al instante el
contenedor **api**, sin reiniciar nada.

Regla del taller: cada entrenamiento crea un `modelo_vN` nuevo. Nunca se pisa una versión anterior.


In [ ]:
from penguins_ml import registry, training

print('Volumen de modelos:', registry.models_dir())
print('Dataset:          ', training.data_path())
print('Versiones actuales:', [v['version'] for v in registry.list_versions()] or 'ninguna')


## 1. Los datos


In [ ]:
df = training.load_data()
print(df.shape)
df.head()


In [ ]:
df['species'].value_counts()


## 2. Entrenar y publicar la primera versión

`train_and_register` entrena los algoritmos que le pidas, calcula métricas sobre el
conjunto de test y publica todo como una versión nueva junto con su `metadata.json`.


In [ ]:
meta = training.train_and_register(
    notas='baseline: los tres algoritmos con parámetros por defecto',
)


Andá a <http://localhost:8025/docs> y abrí `POST /predict`: el desplegable **version**
ya ofrece `modelo_v1`.


## 3. Entrenar una segunda versión

Cambiar hiperparámetros o entrenar un solo algoritmo genera otra versión. La anterior sigue ahí.


In [ ]:
meta_v2 = training.train_and_register(
    ['randomforest'],
    notas='random forest con 400 árboles y profundidad limitada',
    params={'randomforest': {'n_estimators': 400, 'max_depth': 6, 'random_state': 42}},
)


Refrescá `/docs` con F5: ahora el desplegable ofrece `modelo_v1` **y** `modelo_v2`.
La API nunca se reinició.


## 4. El historial completo


In [ ]:
import pandas as pd

filas = [
    {
        'version': v['version'],
        'creado': v['creado'],
        'notas': v['notas'],
        'algoritmos': ', '.join(v['algoritmos']),
        'mejor_accuracy': max((m.get('accuracy', 0) for m in v['modelos'].values()), default=None),
    }
    for v in registry.list_versions()
]
pd.DataFrame(filas)


## 5. Comprobar contra la API

Los dos contenedores están en la misma red de Compose, así que desde acá la API responde
en `http://api:8025`. `version` y `modelo` viajan como parámetros de query.


In [ ]:
import json, urllib.parse, urllib.request

with urllib.request.urlopen('http://api:8025/modelos', timeout=5) as r:
    disponibles = json.load(r)

print('Modelos disponibles según la API:')
for v in disponibles['versiones']:
    algos = ', '.join(a['nombre'] for a in v['algoritmos'])
    print(f"  {v['version']:<12} {algos}")


In [ ]:
MEDIDAS = {
    'bill_length_mm': 39.1,
    'bill_depth_mm': 18.7,
    'flipper_length_mm': 181,
    'body_mass_g': 3750,
    'island': 'Torgersen',
    'sex': 'MALE',
}


def predecir(version='latest', modelo='auto'):
    query = urllib.parse.urlencode({'version': version, 'modelo': modelo})
    req = urllib.request.Request(
        f'http://api:8025/predict?{query}',
        data=json.dumps(MEDIDAS).encode(),
        headers={'Content-Type': 'application/json'},
    )
    with urllib.request.urlopen(req, timeout=10) as r:
        return json.load(r)


predecir()


### El mismo pingüino contra cada versión

Esto es lo que hace el equipo de pruebas: fijar la entrada y variar el modelo.


In [ ]:
comparacion = []
for v in disponibles['versiones']:
    for a in v['algoritmos']:
        r = predecir(v['version'], a['nombre'])
        comparacion.append({
            'version': r['version'],
            'modelo': r['modelo'],
            'species': r['species'],
            'confianza': round(max(r['probabilities'].values()), 4),
        })

pd.DataFrame(comparacion)


---

El equipo de pruebas entra a <http://localhost:8025/docs>, elige la versión en el
desplegable de `POST /predict` y la ejecuta. Cada vez que se corre una celda de
entrenamiento acá, esa lista crece.
